Taking a look into statsbomb data and what kind of information we can extract

In [ ]:
import pandas as pd
import os

event = pd.read_json(
    "../data/raw/events/7483.json"
)

event.head()

event.columns

Index(['id', 'index', 'period', 'timestamp', 'minute', 'second', 'type',
       'possession', 'possession_team', 'play_pattern', 'team', 'duration',
       'tactics', 'related_events', 'player', 'position', 'location', 'pass',
       'under_pressure', 'carry', 'ball_receipt', 'interception', 'dribble',
       'foul_won', 'counterpress', 'shot', 'goalkeeper', 'duel', 'block',
       'ball_recovery', 'foul_committed', 'injury_stoppage', 'substitution',
       'bad_behaviour'],
      dtype='str')

From each event, we can see:

Index(['id', 'index', 'period', 'timestamp', 'minute', 'second', 'type',
       'possession', 'possession_team', 'play_pattern', 'team', 'duration',
       'tactics', 'related_events', 'player', 'position', 'location', 'pass',
       'under_pressure', 'carry', 'ball_receipt', 'interception', 'dribble',
       'foul_won', 'counterpress', 'shot', 'goalkeeper', 'duel', 'block',
       'ball_recovery', 'foul_committed', 'injury_stoppage', 'substitution',
       'bad_behaviour'],
      dtype='str')

In [9]:
lineup = pd.read_json(
    "../data/raw/lineups/7483.json"
)

event.head()

event.columns

Index(['id', 'index', 'period', 'timestamp', 'minute', 'second', 'type',
       'possession', 'possession_team', 'play_pattern', 'team', 'duration',
       'tactics', 'related_events', 'player', 'position', 'location', 'pass',
       'under_pressure', 'carry', 'ball_receipt', 'interception', 'dribble',
       'foul_won', 'counterpress', 'shot', 'goalkeeper', 'duel', 'block',
       'ball_recovery', 'foul_committed', 'injury_stoppage', 'substitution',
       'bad_behaviour'],
      dtype='str')

All lineup information is already found in events

In [ ]:
# what kind of events are we looking at?
event["type"].value_counts()

type
{'id': 30, 'name': 'Pass'}               779
{'id': 42, 'name': 'Ball Receipt*'}      601
{'id': 43, 'name': 'Carry'}              568
{'id': 17, 'name': 'Pressure'}           220
{'id': 2, 'name': 'Ball Recovery'}        88
{'id': 4, 'name': 'Duel'}                 33
{'id': 38, 'name': 'Miscontrol'}          27
{'id': 23, 'name': 'Goal Keeper'}         26
{'id': 22, 'name': 'Foul Committed'}      24
{'id': 21, 'name': 'Foul Won'}            24
{'id': 6, 'name': 'Block'}                24
{'id': 5, 'name': 'Camera On'}            23
{'id': 16, 'name': 'Shot'}                22
{'id': 14, 'name': 'Dribble'}             18
{'id': 10, 'name': 'Interception'}        16
{'id': 9, 'name': 'Clearance'}            16
{'id': 39, 'name': 'Dribbled Past'}       13
{'id': 3, 'name': 'Dispossessed'}         12
{'id': 40, 'name': 'Injury Stoppage'}      6
{'id': 19, 'name': 'Substitution'}         6
{'id': 18, 'name': 'Half Start'}           4
{'id': 34, 'name': 'Half End'}             4
{'id'

From this output, we can see that there are all kinds of events that are tracked. For XG, we are primarily concerned with shot attempts:

In [11]:
# what is specifically seen in the shot event?
event.iloc[0]
event["shot"].dropna().iloc[0]

{'one_on_one': True,
 'statsbomb_xg': 0.29035154,
 'end_location': [117.0, 39.3, 0.5],
 'key_pass_id': 'b7669450-bda3-4c0b-bcc3-4f45241bfef0',
 'type': {'id': 87, 'name': 'Open Play'},
 'technique': {'id': 93, 'name': 'Normal'},
 'outcome': {'id': 100, 'name': 'Saved'},
 'body_part': {'id': 38, 'name': 'Left Foot'},
 'freeze_frame': [{'location': [108.0, 29.0],
   'player': {'id': 5089, 'name': 'Ali Krieger'},
   'position': {'id': 3, 'name': 'Right Center Back'},
   'teammate': False},
  {'location': [115.0, 41.0],
   'player': {'id': 5090, 'name': 'Ashlyn Harris'},
   'position': {'id': 1, 'name': 'Goalkeeper'},
   'teammate': False},
  {'location': [111.0, 37.0],
   'player': {'id': 5071, 'name': 'Monica Hickmann Alves'},
   'position': {'id': 4, 'name': 'Center Back'},
   'teammate': False},
  {'location': [100.0, 51.0],
   'player': {'id': 5063, 'name': 'Chioma Ubogagu'},
   'position': {'id': 16, 'name': 'Left Midfield'},
   'teammate': False},
  {'location': [107.0, 55.0],
   'p

From each shot attempt, we can see lots of important information: where the ball ended, if the shot attempt was one-on-one with the goalie, what technique and what body part was used to shoot, as well as other player information at the time of the shot.

With this data, we'd like to take each event (shot, pass, tackle, etc.) to help build our XGBoost model. Time to do some data processing to turn raw data into processed so we can include only the important information!

This is the most baseline model — as we look to improve accuracy, there are a few features that I'm interested in exploring and adding on:
1. Was this the player's first attempt? (there is the "first_time" variable in the dataset)
2. How many opponents are around him at this time (where "teammate" = false)
3. Shot technique ("technique")
4. Was the player under pressure? ("under_pressure")